SPANDANA UN

NOVEMBER BATCH

25/11/2026

In [ ]:
from datetime import datetime, timedelta
import numpy as np
FILE_CANDIDATES = ["hostel_bois.txt"]
def read_chat_lines(paths):
    for path in paths:
        try:
            with open(path, "r", encoding="utf-8") as f:
                content = f.read()
            return content.split("\n")
        except FileNotFoundError:
            continue
    print("hostel_bois.txt not found,run again.")
    return []
lines = read_chat_lines(FILE_CANDIDATES)
print("Lines read from file:", len(lines))

Lines read from file: 3178


In [ ]:
# AI-assisted: I asked Claude to review my edge-case handling for the parser (continuation lines, BOM, odd system lines).
MONTHS_FULL = ["January", "February", "March", "April", "May", "June", "July",
               "August", "September", "October", "November", "December"]
MONTHS_SHORT = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
MEDIA_TEXT = "<Media omitted>"
DELETED_TEXT = "This message was deleted"
def starts_with_date(line):
    if len(line) < 8:
        return False
    d = line[:8]
    return (d[0:2].isdigit() and d[2] == "/" and d[3:5].isdigit()
            and d[5] == "/" and d[6:8].isdigit())
def parse_chat(lines):
    messages = []
    stats = {"system": 0, "media": 0, "deleted": 0, "group_name": "WhatsApp Group"}
    last_was_message = False
    for raw_line in lines:
        line = raw_line.replace("\ufeff", "").rstrip("\r")
        if line.strip() == "":
            continue
        if not starts_with_date(line):
            if last_was_message:
                messages[-1]["text"] += "\n" + line.strip()
            continue
        if " - " not in line:
            stats["system"] += 1
            last_was_message = False
            continue
        timestamp, rest = line.split(" - ", 1)
        sender, separator, text = rest.partition(": ")
        try:
            moment = datetime.strptime(timestamp, "%d/%m/%y, %H:%M")
        except ValueError:
            moment = None
        if separator == "" or moment is None or len(sender) > 30 or '"' in sender:
            stats["system"] += 1
            last_was_message = False
            if 'created group "' in rest:
                first, last = rest.find('"'), rest.rfind('"')
                if last > first:
                    stats["group_name"] = rest[first + 1:last]
            continue
        text = text.strip()
        is_media = (text == MEDIA_TEXT)
        is_deleted = (text == DELETED_TEXT)
        if is_media:
            stats["media"] += 1
        if is_deleted:
            stats["deleted"] += 1
        messages.append({"timestamp": timestamp, "dt": moment, "sender": sender,
                         "text": text, "is_media": is_media, "is_deleted": is_deleted})
        last_was_message = True
    return messages, stats
def is_real(msg):
    # real text = not a media stub and not a deleted message
    return (not msg["is_media"]) and (not msg["is_deleted"])
def day_start(msg):
    # midnight of the day the message was sent
    return datetime.strptime(msg["timestamp"][:8], "%d/%m/%y")
def chat_span(messages):
    days = [day_start(m) for m in messages]
    first_day, last_day = min(days), max(days)
    return first_day, last_day, (last_day - first_day).days + 1
messages, stats = parse_chat(lines)
first_day, last_day, total_days = chat_span(messages)
participants = set(m["sender"] for m in messages)
print(f"Successfully parsed {len(messages):,} messages from {len(participants)} participants "
      f"over {total_days} days, skipped {stats['system']} system messages "
      f"({stats['media']} media-omitted and {stats['deleted']} deleted messages are counted separately).")
print("\nFirst 5 parsed messages:")
for m in messages[:5]:
    print(f"  {m['timestamp']} | {m['sender']}: {m['text'][:70]}")
print("\nLast 5 parsed messages:")
for m in messages[-5:]:
    print(f"  {m['timestamp']} | {m['sender']}: {m['text'][:70]}")

Successfully parsed 3,174 messages from 6 participants over 60 days, skipped 4 system messages (32 media-omitted and 15 deleted messages are counted separately).

First 5 parsed messages:
  01/04/24, 01:17 | Rahul: scene fix
  01/04/24, 01:17 | Rahul: haan
  01/04/24, 01:18 | Rahul: kya scene
  01/04/24, 02:13 | Rahul: abhi free hai?
  01/04/24, 02:13 | Rahul: abey

Last 5 parsed messages:
  30/05/24, 19:14 | Priya: Take care everyone
  30/05/24, 19:28 | Priya: Karan that sounds tough, take care
  30/05/24, 21:17 | Aman: the existential dread is back
  30/05/24, 21:30 | Karan: Long day guys, woke up at six for that placement workshop which starte
  30/05/24, 23:31 | Aman: anyone awake?


In [ ]:
# AI-assisted: asked Claude about f-string width/alignment tricks for the ranked table.
def count_per_person(messages):
    counts = {}
    for m in messages:
        counts[m["sender"]] = counts.get(m["sender"], 0) + 1
    return counts
def ranked_people(messages):
    return sorted(count_per_person(messages).items(), key=lambda item: item[1], reverse=True)
def format_date(day, month_names=MONTHS_FULL):
    return f"{day.day:02d} {month_names[day.month - 1]} {day.year}"
def per_person_stats(messages, people):
    table = {}
    for p in people:
        mine = [m for m in messages if m["sender"] == p]
        real = [m for m in mine if is_real(m)]
        words = sum(len(m["text"].split()) for m in real)
        table[p] = {
            "messages": len(mine),
            "words": words,
            "avg_words": words / len(real) if real else 0.0,
            "media": sum(1 for m in mine if m["is_media"]),
            "deleted": sum(1 for m in mine if m["is_deleted"]),
        }
    return table
ranked = ranked_people(messages)
people = [name for name, count in ranked]
total_msgs = len(messages)
print("=" * 50)
print(" GROUP OVERVIEW")
print("=" * 50)
print(f" Group : {stats['group_name']}")
print(f" Period : {format_date(first_day)} to {format_date(last_day)} ({total_days} days)")
print(f" Total messages : {total_msgs:,}")
print(f" Participants : {len(people)}")
print()
print(" MESSAGES PER PERSON")
for name, count in ranked:
    print(f"  {name:<8}: {count:>4} ({count / total_msgs * 100:4.1f}%)")
print()
print(" PER-PERSON STATS ")
print(f"  {'Name':<8}{'Msgs':>5}{'Words':>6}{'Avg words':>10}{'Media':>8}{'Deleted':>6}")
table = per_person_stats(messages, people)
for p in people:
    t = table[p]
    print(f"  {p:<8}{t['messages']:>5}{t['words']:>6}{t['avg_words']:>10.1f}{t['media']:>8} {t['deleted']:>6}")

 GROUP OVERVIEW
 Group : Hostel Bois 4ever
 Period : 01 April 2024 to 30 May 2024 (60 days)
 Total messages : 3,174
 Participants : 6

 MESSAGES PER PERSON
  Rahul   :  953 (30.0%)
  Priya   :  718 (22.6%)
  Neha    :  635 (20.0%)
  Aman    :  490 (15.4%)
  Karan   :  354 (11.2%)
  Vikas   :   24 ( 0.8%)

 PER-PERSON STATS 
  Name     Msgs Words Avg words   MediaDeleted
  Rahul     953  2399       2.6       7      6
  Priya     718  3560       5.0       4      2
  Neha      635  3317       5.3       8      3
  Aman      490  2430       5.0       4      2
  Karan     354 19681      57.0       7      2
  Vikas      24    40       1.8       2      0


In [ ]:
def busiest_day(messages):
    per_day = {}
    for m in messages:
        key = m["timestamp"][:8]
        per_day[key] = per_day.get(key, 0) + 1
    best = max(per_day, key=lambda d: per_day[d])
    return best, per_day[best]
def busiest_hour(messages):
    per_hour = {}
    for m in messages:
        per_hour[m["dt"].hour] = per_hour.get(m["dt"].hour, 0) + 1
    best = max(per_hour, key=lambda h: per_hour[h])
    return best, per_hour[best]
day_key, day_count = busiest_day(messages)
day_obj = datetime.strptime(day_key, "%d/%m/%y")
hour, hour_count = busiest_hour(messages)
print(f" Busiest day  : {format_date(day_obj)} ({day_count} messages)")
print(f" Busiest hour : {hour:02d}:00 - {(hour + 1) % 24:02d}:00 "
      f"({hour_count} messages in total, avg {hour_count / total_days:.1f} per day)")

 Busiest day  : 04 May 2024 (76 messages)
 Busiest hour : 18:00 - 19:00 (248 messages in total, avg 4.1 per day)


In [ ]:
# AI-assisted: I asked Claude for help choosing the shading thresholds and header alignment.
def build_heatmap(messages, people):
    row_of = {}
    for i, p in enumerate(people):
        row_of[p] = i
    matrix = np.zeros((len(people), 24), dtype=int)
    for m in messages:
        matrix[row_of[m["sender"]], m["dt"].hour] += 1
    return matrix
def shade(value, row_max):
    if row_max == 0:
        return ". "
    share = value / row_max
    if share <= 0.50:
        return ". "
    elif share <= 0.75:
        return "░ "
    elif share < 1.00:
        return "▒ "
    return "█ "
def night_share(matrix):
    night = matrix[:, 23] + matrix[:, 0:5].sum(axis=1)
    totals = matrix.sum(axis=1)
    return night / np.where(totals == 0, 1, totals)
def render_heatmap(matrix, people, tags=None):
    tags = tags or {}
    name_w = max(len(p) for p in people)
    header = "".join((f"{h:02d}" if h % 3 == 0 else "  ") for h in range(24))
    print("  " + " " * name_w + "  " + header)
    for i, p in enumerate(people):
        row_max = matrix[i].max()
        cells = "".join(shade(matrix[i, h], row_max) for h in range(24))
        tag = f" <- {tags[p]}" if p in tags else ""
        print(f"  {p:<{name_w}}  {cells}{tag}")
heatmap = build_heatmap(messages, people)
render_heatmap(heatmap, people)
print("\n  . = 0-50%   ░ = 50-75%   ▒ = 75-99%   █ = 100% ")
print("\nShape of matrix:", heatmap.shape)
print("Row totals match message counts:", list(heatmap.sum(axis=1)) == [count for name, count in ranked])
shares = night_share(heatmap)
for i, p in enumerate(people):
    print(f"  {p:<8} night share (23h-04h): {shares[i] * 100:5.1f}%")

         00    03    06    09    12    15    18    21    
  Rahul  . . . . . . . . . . . . ░ . . ░ ░ . █ ░ . ▒ ░ ░ 
  Priya  . . . . . . . . ░ █ ▒ ▒ ▒ ░ ░ . . ░ ░ ▒ ░ . . . 
  Neha   . . . . . . . . ░ ▒ ▒ . ░ ░ . . ░ ▒ █ ▒ ░ . . . 
  Aman   ░ ▒ ░ ░ █ . . . . . . . . . . . . . . . . . . ░ 
  Karan  . . . . . . . . . . ░ . █ ░ ▒ ░ ░ ░ ░ ▒ ░ . . . 
  Vikas  . . . . . . . . █ . . . ░ ░ . . . █ ░ ░ . . . ░ 

  . = 0-50%   ░ = 50-75%   ▒ = 75-99%   █ = 100% 

Shape of matrix: (6, 24)
Row totals match message counts: True
  Rahul    night share (23h-04h):  13.4%
  Priya    night share (23h-04h):   1.3%
  Neha     night share (23h-04h):   4.7%
  Aman     night share (23h-04h):  79.8%
  Karan    night share (23h-04h):   2.3%
  Vikas    night share (23h-04h):   8.3%


In [ ]:
PUNCTUATION = ".,!?;:\"()[]{}*_~-/\\"
STOP_WORDS = {
    "i", "is", "the", "a", "an", "and", "or", "to", "of", "in", "on", "for", "at", "by",
    "it", "its", "this", "that", "these", "those", "was", "were", "be", "been", "am", "are",
    "as", "but", "if", "so", "my", "me", "you", "your", "we", "he", "she", "his", "her",
    "they", "them", "have", "has", "had", "do", "did", "does", "not", "no", "with", "from",
    "about", "just", "up", "what", "how", "which", "one", "there", "here", "can", "will",
    "would", "all", "also", "then", "than", "too", "very", "i'm", "i'll", "it's", "don't",
    "can't", "today","everyone", "anyone", "telling", "started", "entire", "everything", "came", "please", "now",
    "who", "why", "used", "three", "way", "most", "something", "off", "day",
    # very common Hindi/Hinglish fillers (they play the role of 'is', 'the', 'and')
    "hai", "hain", "ho", "hu", "tha", "thi", "ka", "ki", "ke", "ko", "se", "ne", "na",
    "ek", "bhi", "aur", "toh", "ja", "raha", "kar", "haan", "aa",
}
def tokenize(text):
    words = []
    for w in text.lower().replace("’", "'").split():
        w = w.strip(PUNCTUATION)
        if w != "":
            words.append(w)
    return words
def word_frequency(msgs):
    freq = {}
    for m in msgs:
        if not is_real(m):
            continue
        for w in tokenize(m["text"]):
            if w in STOP_WORDS or w.isdigit():
                continue
            freq[w] = freq.get(w, 0) + 1
    return freq
def top_n(freq, n):
    return sorted(freq.items(), key=lambda item: (-item[1], item[0]))[:n]
def make_bar(value, biggest, width=30):
    if biggest == 0:
        return "."
    length = round(value / biggest * width)
    return "█" * length if length > 0 else "."
group_top = top_n(word_frequency(messages), 10)
print(" THIS GROUP'S FAVOURITE WORDS")
for word, count in group_top:
    print(f"  {word:<9} {make_bar(count, group_top[0][1]):<20} {count}")
print("\n TOP 5 WORDS PER PERSON")
for p in people:
    mine = [m for m in messages if m["sender"] == p]
    words = ", ".join(f"{w} ({c})" for w, c in top_n(word_frequency(mine), 6))
    print(f"  {p:<9}: {words}")

 THIS GROUP'S FAVOURITE WORDS
  guys      ██████████████████████████████ 318
  bhai      ███████████████      160
  scene     ██████████████       145
  yaar      █████████████        139
  kya       █████████████        133
  sleep     ███████████          112
  life      █████████            94
  aman      █████████            93
  okay      █████████            92
  ok        ████████             85

 TOP 5 WORDS PER PERSON
  Rahul    : bhai (159), scene (144), kya (133), yaar (105), abey (58), karte (40)
  Priya    : aman (93), okay (80), take (72), guys (65), eat (60), care (58)
  Neha     : guys (101), cant (58), ok (52), wait (48), believe (47), literally (40)
  Aman     : sleep (71), wonder (43), night (41), guys (32), thinking (32), hits (31)
  Karan    : guys (120), because (70), book (70), completely (70), ended (69), wild (68)
  Vikas    : haha (4), busy (3), sorry (3), exam (2), sun (2), alag (1)


In [ ]:
def response_gaps(messages):
    gaps = {}
    for i in range(1, len(messages)):
        previous, current = messages[i - 1], messages[i]
        if current["sender"] != previous["sender"]:
            seconds = (current["dt"] - previous["dt"]).total_seconds()
            if current["sender"] not in gaps:
                gaps[current["sender"]] = []
            gaps[current["sender"]].append(seconds)
    return gaps
def average_response_seconds(messages):
    gaps = response_gaps(messages)
    return {p: sum(g) / len(g) for p, g in gaps.items()}
def format_duration(seconds):
    if seconds < 3600:
        return f"{seconds / 60:.1f} minutes"
    return f"{seconds / 3600:.1f} hours"
def longest_silent_streaks(messages, first_day, total_days):
    active = {}
    for m in messages:
        index = (day_start(m) - first_day).days
        if m["sender"] not in active:
            active[m["sender"]] = set()
        active[m["sender"]].add(index)
    result = {}
    for person, days in active.items():
        best_len, best_start = 0, 0
        current_len, current_start = 0, 0
        for d in range(total_days):
            if d in days:
                current_len = 0
            else:
                if current_len == 0:
                    current_start = d
                current_len += 1
                if current_len > best_len:
                    best_len, best_start = current_len, current_start
        result[person] = (best_len, best_start)
    return result
def streak_range(first_day, start, length, joiner="to"):
    s = first_day + timedelta(days=start)
    e = first_day + timedelta(days=start + length - 1)
    start_text = f"{s.day:02d} {MONTHS_SHORT[s.month - 1]}"
    end_text = f"{e.day:02d} {MONTHS_SHORT[e.month - 1]}"
    return start_text if length == 1 else f"{start_text} {joiner} {end_text}"
averages = average_response_seconds(messages)
fast = min(averages, key=lambda p: averages[p])
slow = max(averages, key=lambda p: averages[p])
print(" RESPONSE PATTERNS")
print(f"  Fastest replier : {fast} (avg {format_duration(averages[fast])})")
print(f"  Slowest replier : {slow} (avg {format_duration(averages[slow])})")
print("\n LONGEST SILENT STREAKS")
streaks = longest_silent_streaks(messages, first_day, total_days)
for person in sorted(streaks, key=lambda p: -streaks[p][0]):
    length, start = streaks[person]
    if length == 0:
        print(f"  {person:<8}: 0 days (never went silent)")
    else:
        word = "day" if length == 1 else "days"
        print(f"  {person:<9}: {length} {word} ({streak_range(first_day, start, length)})")

 RESPONSE PATTERNS
  Fastest replier : Rahul (avg 34.9 minutes)
  Slowest replier : Aman (avg 55.4 minutes)

 LONGEST SILENT STREAKS
  Vikas    : 11 days (23 Apr to 03 May)
  Rahul   : 0 days (never went silent)
  Priya   : 0 days (never went silent)
  Karan   : 0 days (never went silent)
  Neha    : 0 days (never went silent)
  Aman    : 0 days (never went silent)


In [ ]:
MIN_REAL_MESSAGES = 50
CARING_PHRASES = ["okay", "safe", "eat", "sleep", "take care", "are you", "please",
                  "reminder", "drink water", "don't forget"]
LAUGH_WORDS = {"lol", "lmao", "haha", "rofl", "lmfao"}
DEEP_WORDS = {"life", "meaning", "exist", "existential", "existence", "dread",
              "universe", "purpose", "wonder", "wondering"}
def real_of(person, messages):
    return [m for m in messages if m["sender"] == person and is_real(m)]
def metric_spammer(person, messages, total_days):
    runs, current_sender, run_length = [], None, 0
    for m in messages:
        if m["sender"] == current_sender:
            run_length += 1
        else:
            if current_sender == person:
                runs.append(run_length)
            current_sender, run_length = m["sender"], 1
    if current_sender == person:
        runs.append(run_length)
    return sum(runs) / len(runs) if runs else 0.0
def metric_group_mom(person, messages, total_days):
    mine = real_of(person, messages)
    if len(mine) < MIN_REAL_MESSAGES:
        return 0.0
    hits = 0
    for m in mine:
        padded = " " + " ".join(tokenize(m["text"])) + " "
        for phrase in CARING_PHRASES:
            hits += padded.count(" " + phrase + " ")
    return float(hits)
def metric_night_owl(person, messages, total_days):
    mine = [m for m in messages if m["sender"] == person]
    if not mine:
        return 0.0
    night = sum(1 for m in mine if m["dt"].hour >= 23 or m["dt"].hour <= 4)
    return night / len(mine)
def metric_storyteller(person, messages, total_days):
    mine = real_of(person, messages)
    if len(mine) < MIN_REAL_MESSAGES:
        return 0.0
    return sum(len(m["text"].split()) for m in mine) / len(mine)
def metric_drama_queen(person, messages, total_days):
    mine = real_of(person, messages)
    if len(mine) < MIN_REAL_MESSAGES:
        return 0.0
    dramatic = 0
    for m in mine:
        text = m["text"].strip()
        shouting = len(text) >= 2 and text.isupper()
        if shouting or text.count("!") >= 2:
            dramatic += 1
    return dramatic / len(mine)
def metric_ghost(person, messages, total_days):
    mine = [m for m in messages if m["sender"] == person]
    active_days = set(m["timestamp"][:8] for m in mine)
    return (total_days - len(active_days)) / total_days if total_days > 0 else 0.0
def metric_comedian(person, messages, total_days):
    mine = real_of(person, messages)
    if len(mine) < MIN_REAL_MESSAGES:
        return 0.0
    funny = 0
    for m in mine:
        for w in tokenize(m["text"]):
            if w in LAUGH_WORDS or w.startswith("haha"):
                funny += 1
                break
    return funny / len(mine)
def metric_question_master(person, messages, total_days):
    mine = real_of(person, messages)
    if len(mine) < MIN_REAL_MESSAGES:
        return 0.0
    return sum(1 for m in mine if m["text"].strip().endswith("?")) / len(mine)
def metric_philosopher(person, messages, total_days):
    after_midnight = [m for m in real_of(person, messages) if m["dt"].hour <= 4]
    if len(after_midnight) < 20:
        return 0.0
    deep = 0
    for m in after_midnight:
        for w in tokenize(m["text"]):
            if w in DEEP_WORDS:
                deep += 1
                break
    return deep / len(after_midnight)
ARCHETYPES = [
    ("THE SPAMMER", metric_spammer, 3.0, False,
     lambda raw, days: f"avg {raw:.1f} msgs in a row"),
    ("THE GROUP MOM", metric_group_mom, 100, True,
     lambda raw, days: f"caring keyword score: {int(raw)}"),
    ("THE NIGHT OWL", metric_night_owl, 0.60, False,
     lambda raw, days: f"{raw * 100:.1f}% msgs between 23h-04h"),
    ("THE STORYTELLER", metric_storyteller, 30, False,
     lambda raw, days: f"avg {raw:.1f} words per msg"),
    ("THE DRAMA QUEEN", metric_drama_queen, 0.30, False,
     lambda raw, days: f"{raw * 100:.1f}% ALL-CAPS / !! messages"),
    ("THE GHOST", metric_ghost, 0.60, False,
     lambda raw, days: f"silent on {round(raw * days)} of {days} days"),
    ("THE COMEDIAN", metric_comedian, 0.10, True,
     lambda raw, days: f"{raw * 100:.1f}% laugh messages"),
    ("THE QUESTION MASTER", metric_question_master, 0.25, False,
     lambda raw, days: f"{raw * 100:.1f}% msgs end with ?"),
    ("THE LATE-NIGHT PHILOSOPHER", metric_philosopher, 0.18, False,
     lambda raw, days: f"{raw * 100:.0f}% of after-midnight msgs get deep"),
]
def compute_scores(people, messages, total_days):
    raw = {p: {} for p in people}
    scores = {p: {} for p in people}
    for title, metric, threshold, leader_only, describe in ARCHETYPES:
        for p in people:
            raw[p][title] = metric(p, messages, total_days)
        leader = max(people, key=lambda p: raw[p][title]) if leader_only else None
        for p in people:
            if leader_only and p != leader:
                scores[p][title] = 0.0
            else:
                scores[p][title] = raw[p][title] / threshold
    return raw, scores
def assign_archetypes(people, scores):
    pairs = []
    for p in people:
        for title in scores[p]:
            pairs.append((scores[p][title], p, title))
    pairs.sort(key=lambda item: (-item[0], item[1], item[2]))
    assigned, taken = {}, set()
    for score, p, title in pairs:
        if p not in assigned and title not in taken:
            assigned[p] = title
            taken.add(title)
    for p in people:
        if p not in assigned:
            assigned[p] = max(scores[p], key=lambda t: scores[p][t])
    return assigned
def runner_up(person, assigned_title, scores):
    options = {t: s for t, s in scores[person].items() if t != assigned_title}
    best = max(options, key=lambda t: options[t]) if options else None
    if best is not None and options[best] >= 1.0:
        return best
    return None
raw_scores, scores = compute_scores(people, messages, total_days)
assigned = assign_archetypes(people, scores)
describers = {title: describe for title, metric, threshold, leader_only, describe in ARCHETYPES}
print(" PERSONALITY ARCHETYPES")
for p in people:
    title = assigned[p]
    detail = describers[title](raw_scores[p][title], total_days)
    print(f"  {p:<6} → {title} ({detail})")
    second = runner_up(p, title, scores)
    if second:
        print(f"  {'':<7}   runner-up: {second} ({describers[second](raw_scores[p][second], total_days)})")

 PERSONALITY ARCHETYPES
  Rahul  → THE SPAMMER (avg 4.5 msgs in a row)
  Priya  → THE GROUP MOM (caring keyword score: 576)
            runner-up: THE QUESTION MASTER (29.5% msgs end with ?)
  Neha   → THE DRAMA QUEEN (63.3% ALL-CAPS / !! messages)
  Aman   → THE NIGHT OWL (79.8% msgs between 23h-04h)
            runner-up: THE LATE-NIGHT PHILOSOPHER (20% of after-midnight msgs get deep)
  Karan  → THE STORYTELLER (avg 57.0 words per msg)
  Vikas  → THE GHOST (silent on 44 of 60 days)


In [ ]:
# AI-assisted: Claude helped me polish the layout and spacing of the report.
WIDTH = 60
def section(title):
    print()
    print(f" {title}")
    print(" " + "─" * (WIDTH - 2))
def build_report(messages, stats):
    if len(messages) == 0:
        print("No messages found, nothing to analyse. Check the file path and the export format.")
        return
    first_day, last_day, total_days = chat_span(messages)
    ranked = ranked_people(messages)
    people = [name for name, count in ranked]
    total = len(messages)
    name_w = max(len(p) for p in people)
    raw_scores, scores = compute_scores(people, messages, total_days)
    assigned = assign_archetypes(people, scores)
    describers = {title: describe for title, metric, threshold, leader_only, describe in ARCHETYPES}
    print("=" * WIDTH)
    print(f"{'GROUPDNA REPORT — ' + chr(34) + stats['group_name'] + chr(34):^{WIDTH}}")
    member_word = "member" if len(people) == 1 else "members"
    subtitle = f"{total_days} days • {total:,} messages • {len(people)} {member_word}"
    print(f"{subtitle:^{WIDTH}}")
    print("=" * WIDTH)
    day_key, day_count = busiest_day(messages)
    hour, hour_count = busiest_hour(messages)
    print(f" Period      : {format_date(first_day)} to {format_date(last_day)}")
    print(f" Busiest day : {format_date(datetime.strptime(day_key, '%d/%m/%y'))} ({day_count} messages)")
    print(f" Busiest hour: {hour:02d}:00 - {(hour + 1) % 24:02d}:00 (avg {hour_count / total_days:.1f} messages per day)")
    section("MESSAGES PER PERSON")
    for name, count in ranked:
        print(f"  {name:<{name_w}}  {make_bar(count, ranked[0][1]):<20} {count:>4} ({count / total * 100:4.1f}%)")
    section("MEDIA & DELETED")
    table = per_person_stats(messages, people)
    most_media = max(people, key=lambda p: table[p]["media"])
    most_deleted = max(people, key=lambda p: table[p]["deleted"])
    print(f"  Media shared in total : {stats['media']} (most by {most_media}: {table[most_media]['media']})")
    print(f"  Deleted messages      : {stats['deleted']} (most by {most_deleted}: {table[most_deleted]['deleted']})")
    print(f"  System messages       : {stats['system']}")
    section("ACTIVITY HEATMAP (hour of day, columns 00 to 23)")
    heatmap = build_heatmap(messages, people)
    tags = {p: "NIGHT OWL" for p in people if assigned[p] == "THE NIGHT OWL"}
    render_heatmap(heatmap, people, tags)
    section("THIS GROUP'S FAVOURITE WORDS")
    top = top_n(word_frequency(messages), 5)
    for word, count in top:
        print(f"  {word:<10} {make_bar(count, top[0][1]):<20} {count}")
    section("RESPONSE PATTERNS")
    averages = average_response_seconds(messages)
    if averages:
        fastest = min(averages, key=lambda p: averages[p])
        slowest = max(averages, key=lambda p: averages[p])
        print(f"  Fastest replier : {fastest} (avg {format_duration(averages[fastest])})")
        print(f"  Slowest replier : {slowest} (avg {format_duration(averages[slowest])})")
    else:
        print("  Not enough back-and-forth to measure response times.")
    section("LONGEST SILENT STREAKS")
    streaks = longest_silent_streaks(messages, first_day, total_days)
    for person in sorted(streaks, key=lambda p: (-streaks[p][0], p)):
        length, start = streaks[person]
        if length == 0:
            print(f"  {person:<{name_w}} : 0 days (never went silent)")
        else:
            word = "day" if length == 1 else "days"
            print(f"  {person:<{name_w}} : {length} {word} ({streak_range(first_day, start, length, '-')})")
    section("PERSONALITY ARCHETYPES")
    for p in people:
        title = assigned[p]
        detail = describers[title](raw_scores[p][title], total_days)
        print(f"  {p:<{name_w}} → {title} ({detail})")
        second = runner_up(p, title, scores)
        if second:
            print(f"  {'':<{name_w}}   ↳ runner-up: {second}")
    print()
    print("=" * WIDTH)
    print(f"{'Generated by GroupDNA • Built with Python + NumPy':^{WIDTH}}")
    print("=" * WIDTH)
build_report(messages, stats)

           GROUPDNA REPORT — "Hostel Bois 4ever"            
            60 days • 3,174 messages • 6 members            
 Period      : 01 April 2024 to 30 May 2024
 Busiest day : 04 May 2024 (76 messages)
 Busiest hour: 18:00 - 19:00 (avg 4.1 messages per day)

 MESSAGES PER PERSON
 ──────────────────────────────────────────────────────────
  Rahul  ██████████████████████████████  953 (30.0%)
  Priya  ███████████████████████  718 (22.6%)
  Neha   ████████████████████  635 (20.0%)
  Aman   ███████████████       490 (15.4%)
  Karan  ███████████           354 (11.2%)
  Vikas  █                      24 ( 0.8%)

 MEDIA & DELETED
 ──────────────────────────────────────────────────────────
  Media shared in total : 32 (most by Neha: 8)
  Deleted messages      : 15 (most by Rahul: 6)
  System messages       : 4

 ACTIVITY HEATMAP (hour of day, columns 00 to 23)
 ──────────────────────────────────────────────────────────
         00    03    06    09    12    15    18    21    
  Rahul  . . .

In [ ]:
print("TEST 1: EMPTY FILE")
empty_messages, empty_stats = parse_chat([])
build_report(empty_messages, empty_stats)

print("\n\nTEST 2: ONE PERSON (with a system line and a multi-line message)")
one_person_lines = [
    '01/04/24, 10:00 - rahul created group "Solo Test"',
    "01/04/24, 10:05 - rahul: hello? anyone here",
    "02/04/24, 22:10 - rahul: this is a long message",
    "that continues on a second line",
    "02/04/24, 22:11 - rahul: <Media omitted>",
    "",
    "04/04/24, 09:30 - rahul: This message was deleted",
]
one_messages, one_stats = parse_chat(one_person_lines)
print("Parsed:", len(one_messages), "messages | multi-line text ->", repr(one_messages[1]["text"]))
build_report(one_messages, one_stats)

TEST 1: EMPTY FILE
No messages found, nothing to analyse. Check the file path and the export format.


TEST 2: ONE PERSON (with a system line and a multi-line message)
Parsed: 4 messages | multi-line text -> 'this is a long message\nthat continues on a second line'
               GROUPDNA REPORT — "Solo Test"                
               4 days • 4 messages • 1 member               
 Period      : 01 April 2024 to 04 April 2024
 Busiest day : 02 April 2024 (2 messages)
 Busiest hour: 22:00 - 23:00 (avg 0.5 messages per day)

 MESSAGES PER PERSON
 ──────────────────────────────────────────────────────────
  rahul  ██████████████████████████████    4 (100.0%)

 MEDIA & DELETED
 ──────────────────────────────────────────────────────────
  Media shared in total : 1 (most by rahul: 1)
  Deleted messages      : 1 (most by rahul: 1)
  System messages       : 1

 ACTIVITY HEATMAP (hour of day, columns 00 to 23)
 ──────────────────────────────────────────────────────────
         00    03    